In [1]:
import cv2 as cv
import numpy as np
import pandas as pd
import os
import time
from video_to_frames import video_to_frames

Started processing of /home/eirik-holm/Documents/Bachelor/fpga_neural_network/Python_implementation/video.mp4


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

Success! Frames are in: /home/eirik-holm/Documents/Bachelor/fpga_neural_network/Python_implementation/Images


[out#0/image2 @ 0x55fb12b28100] video:46794kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: unknown
frame=  351 fps=185 q=-0.0 Lsize=N/A time=00:00:35.00 bitrate=N/A speed=18.4x    


**Parameters**

In [2]:
# Frame params
FRAME_HEIGHT = 480
FRAME_WIDTH = 640

num_bins_x = 4
num_bins_y = 3

# Keypoint params
max_kpoints = 500
fast_threshold = 15

# Number of frames to process
n_frames = 50

In [3]:
# Get the directory where the script is located
script_dir = os.getcwd()

# Create output folder
output_folder = os.path.join(script_dir, "fast_keypoints")
os.makedirs(output_folder, exist_ok=True)

# Initiate FAST detector
fast = cv.FastFeatureDetector_create(threshold=fast_threshold)
fast.setNonmaxSuppression(True)

# Storing keypoint data
kp_list = []

# Process images
for i in range(1, n_frames + 1):
    img_path = os.path.join(script_dir, "Images", f"img{i:04d}.png")
    
    img = cv.imread(img_path, cv.IMREAD_GRAYSCALE)
    
    if img is None:
        print(f"Failed to load {img_path}")
        continue
    
    # Detect keypoints
    kp = fast.detect(img, None)
    
    # Limit to max_kpoints if specified
    if len(kp) > max_kpoints:
        kp = sorted(kp, key=lambda x: x.response, reverse=True)[:max_kpoints]

    # Save raw keypoints
    kp_list.append(kp)
    
    # Draw keypoints
    img_kp = cv.drawKeypoints(img, kp, None, color=(0,255,0), flags=0)
    
    # Draw grid
    bin_width = FRAME_WIDTH // num_bins_x
    bin_height = FRAME_HEIGHT // num_bins_y
    
    # Vertical lines
    for x in range(0, FRAME_WIDTH + 1, bin_width):
        cv.line(img_kp, (x, 0), (x, FRAME_HEIGHT), (0, 0, 255), 2)
    
    # Horizontal lines
    for y in range(0, FRAME_HEIGHT + 1, bin_height):
        cv.line(img_kp, (0, y), (FRAME_WIDTH, y), (0, 0, 255), 2)
    
    # Save result
    output_path = os.path.join(output_folder, f'kp_img{i:04d}.png')
    cv.imwrite(output_path, img_kp)

print(f"\nAll images saved to: {output_folder}")


All images saved to: /home/eirik-holm/Documents/Bachelor/fpga_neural_network/Python_implementation/fast_keypoints


**Bin point count**

In [4]:
x_data = []
y_data = []
for frame in kp_list:
    x_data.extend([kp.pt[0] for kp in frame if kp.response >= fast_threshold])
    y_data.extend([kp.pt[1] for kp in frame if kp.response >= fast_threshold])

hist, x_edges, y_edges = np.histogram2d(x=x_data, y=y_data, bins=[num_bins_x, num_bins_y], range=[[0, FRAME_HEIGHT], [0, FRAME_WIDTH]])

print("Total point count per cell:")

for i in range(len(x_edges) - 1):
    for j in range(len(y_edges) - 1):
        print(f"Point count [[{x_edges[i]:.2f}, {x_edges[i+1]:.2f}], \t [{y_edges[j]:.2f}, {y_edges[j+1]:.2f}]]: \t{hist[i, j]}")


Total point count per cell:
Point count [[0.00, 120.00], 	 [0.00, 213.33]]: 	2285.0
Point count [[0.00, 120.00], 	 [213.33, 426.67]]: 	1448.0
Point count [[0.00, 120.00], 	 [426.67, 640.00]]: 	260.0
Point count [[120.00, 240.00], 	 [0.00, 213.33]]: 	1550.0
Point count [[120.00, 240.00], 	 [213.33, 426.67]]: 	2017.0
Point count [[120.00, 240.00], 	 [426.67, 640.00]]: 	348.0
Point count [[240.00, 360.00], 	 [0.00, 213.33]]: 	1043.0
Point count [[240.00, 360.00], 	 [213.33, 426.67]]: 	704.0
Point count [[240.00, 360.00], 	 [426.67, 640.00]]: 	162.0
Point count [[360.00, 480.00], 	 [0.00, 213.33]]: 	1665.0
Point count [[360.00, 480.00], 	 [213.33, 426.67]]: 	538.0
Point count [[360.00, 480.00], 	 [426.67, 640.00]]: 	55.0


In [5]:
from frame_processing_and_encoding import FrameProcessor

####### Testing ########
script_dir = os.getcwd()
start = time.perf_counter()
spike_trains = []

# Initialize processor
thresholds = (np.linspace(0, 249, 50))
processor = FrameProcessor(threshold_edges=thresholds, n_bins_x=num_bins_x, n_bins_y=num_bins_y)

# Loop through n_frame images
for idx in range(1, n_frames+1):
    img_path = os.path.join(script_dir, "Images", f"img{idx:04d}.png")
    
    # Returns spike train per frame
    spike_train = processor.process_and_encode_frame(img_path)
    spike_trains.append(spike_train)
    
    #print(f"Frame {idx:04d} results:")
    print(f"Spike Train: {spike_train}")
    #print("-" * 30)

end = time.perf_counter()
avg_time_per_frame = (end - start)/n_frames
print(f'Processing time per frame: {avg_time_per_frame:.6f} seconds')

df_results = pd.DataFrame(spike_trains)
df_results.to_csv(os.path.join(script_dir, "Frame_test_spiketrains.csv"), index=False)

Spike Train: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Spike Train: [0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0]
Spike Train: [1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]
Spike Train: [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0]
Spike Train: [1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0]
Spike Train: [1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0]
Spike Train: [1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0]
Spike Train: [0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0]
Spike Train: [1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
Spike Train: [1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0]
Spike Train: [1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0]
Spike Train: [1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0]
Spike Train: [1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0]
Spike Train: [1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0]
Spike Train: [1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0]
Spike Train: [1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0]
Spike Train: [0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
Spike Train: [0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0]
Spike Train: [1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0]
Spike Train: [0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0]


In [6]:
print(thresholds)

[  0.           5.08163265  10.16326531  15.24489796  20.32653061
  25.40816327  30.48979592  35.57142857  40.65306122  45.73469388
  50.81632653  55.89795918  60.97959184  66.06122449  71.14285714
  76.2244898   81.30612245  86.3877551   91.46938776  96.55102041
 101.63265306 106.71428571 111.79591837 116.87755102 121.95918367
 127.04081633 132.12244898 137.20408163 142.28571429 147.36734694
 152.44897959 157.53061224 162.6122449  167.69387755 172.7755102
 177.85714286 182.93877551 188.02040816 193.10204082 198.18367347
 203.26530612 208.34693878 213.42857143 218.51020408 223.59183673
 228.67346939 233.75510204 238.83673469 243.91836735 249.        ]
